# 01 EDA - Airbnb Price per City

This notebook explores the raw Airbnb dataset, checks price behavior, reviews practical variables for modeling, and creates the processed dataset used by the modeling notebook.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

project_dir = Path('/Users/andreyvargassolis/Desktop/Data_Science/AirBnB_PricePerCity')
if str(project_dir / 'src') not in sys.path:
    sys.path.append(str(project_dir / 'src'))

from preprocessing import (
    DISTRICT_MIN_COUNT,
    EXCLUDED_FEATURE_GROUPS,
    MODEL_FEATURES,
    build_model_dataframe,
    get_outlier_bounds,
    load_raw_data,
    remove_price_outliers,
    save_processed_data,
)

sns.set_theme(style='whitegrid')

In [ ]:
raw_df = load_raw_data()

print(raw_df.info())
raw_df.head()

## Dataset Overview

In [ ]:
display(raw_df.describe(include='all').T)
print(f'Rows: {raw_df.shape[0]}')
print(f'Columns: {raw_df.shape[1]}')
print(f'Duplicated rows: {raw_df.duplicated().sum()}')

## Exploratory Market Analysis

In [ ]:
# Prepare a filtered EDA dataframe for clearer visualizations.
lower_bound, upper_bound = get_outlier_bounds(raw_df)
eda_df = remove_price_outliers(raw_df)

price_summary = raw_df['price_total'].describe()
review_columns = [col for col in ['guest_satisfaction_score', 'cleanliness_score'] if col in eda_df.columns]
numeric_correlations = (
    eda_df.select_dtypes(include=np.number)
    .corr(numeric_only=True)['price_total']
    .drop('price_total')
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

print(f'Rows in raw dataset: {len(raw_df)}')
print(f'Rows after IQR price filter for EDA plots: {len(eda_df)}')
print(f'Price lower bound: {lower_bound:.2f}')
print(f'Price upper bound: {upper_bound:.2f}')

### Price Distribution

In [ ]:
plt.figure(figsize=(11, 5))
sns.histplot(data=eda_df, x='price_total', bins=50, kde=True, color='steelblue')
plt.title('Price distribution without IQR outliers')
plt.xlabel('price_total')
plt.ylabel('Listings')
plt.tight_layout()
plt.show()

display(price_summary.to_frame(name='price_total'))

print('Interpretation:')
print(f"- The raw average price is {price_summary['mean']:.2f}, while the median is {price_summary['50%']:.2f}.")
print('- The mean is higher than the median, which suggests a right-skewed distribution: most listings are cheaper, but some expensive listings pull the average upward.')
print('- The plot uses the IQR-filtered dataframe only to make the distribution easier to read visually.')

### Average Price By Neighborhood

In [ ]:
district_price = (
    eda_df.groupby('district')['price_total']
    .agg(['mean', 'median', 'count'])
    .query('count >= 30')
    .sort_values('mean', ascending=False)
    .head(15)
)

plt.figure(figsize=(11, 7))
sns.barplot(data=district_price.reset_index(), x='mean', y='district', color='darkseagreen')
plt.title('Top 15 neighborhoods by average price')
plt.xlabel('Average price_total')
plt.ylabel('district')
plt.tight_layout()
plt.show()

display(district_price)

top_district = district_price.index[0]
top_district_mean = district_price.iloc[0]['mean']

print('Interpretation:')
print(f'- Among neighborhoods with at least 30 listings, {top_district} has the highest average price: {top_district_mean:.2f}.')
print('- Neighborhood clearly matters for price. This supports keeping location-related variables such as city and district in the modeling notebook.')
print('- The minimum count filter avoids ranking neighborhoods with too few listings.')

### Review Scores And Price

In [ ]:
review_correlations = (
    eda_df[review_columns + ['price_total']]
    .corr(numeric_only=True)['price_total']
    .drop('price_total')
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

fig, axes = plt.subplots(1, len(review_columns), figsize=(7 * len(review_columns), 5), squeeze=False)

for axis, column in zip(axes[0], review_columns):
    sns.scatterplot(data=eda_df.sample(min(8000, len(eda_df)), random_state=42), x=column, y='price_total', alpha=0.25, ax=axis)
    axis.set_title(f'{column} vs price_total')
    axis.set_xlabel(column)
    axis.set_ylabel('price_total')

plt.tight_layout()
plt.show()

display(review_correlations.to_frame('pearson_corr_with_price'))

strongest_review_corr = review_correlations.index[0]
strongest_review_corr_value = review_correlations.iloc[0]

print('Interpretation:')
print('- The dataset does not include review counts, so guest_satisfaction_score and cleanliness_score are used as quality/review proxies.')
print(f'- The strongest review-related correlation is {strongest_review_corr}: {strongest_review_corr_value:.3f}.')
print('- The relationship is weak, so review scores alone do not explain price strongly. These variables are also not ideal for pre-listing prediction because a new Airbnb would not know them yet.')

### Price By Room Type

In [ ]:
room_summary = (
    eda_df.groupby('room_type')['price_total']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
)

plt.figure(figsize=(9, 5))
sns.boxplot(data=eda_df, x='room_type', y='price_total', order=room_summary.index)
plt.title('Price by room type')
plt.xlabel('room_type')
plt.ylabel('price_total')
plt.tight_layout()
plt.show()

display(room_summary)

top_room_type = room_summary.index[0]
top_room_mean = room_summary.iloc[0]['mean']

print('Interpretation:')
print(f'- {top_room_type} has the highest average price: {top_room_mean:.2f}.')
print('- This matches the intuition that entire homes/apartments tend to cost more than private or shared rooms.')
print('- Because room_type explains meaningful price differences, it is kept as a modeling feature.')

### Listings Map By Location

In [ ]:
sample_df = eda_df.sample(min(8000, len(eda_df)), random_state=42)

plt.figure(figsize=(10, 7))
location_plot = plt.scatter(
    sample_df['longitude'],
    sample_df['latitude'],
    c=sample_df['price_total'],
    cmap='viridis',
    s=8,
    alpha=0.6,
)
plt.title('Listings by location and price')
plt.xlabel('longitude')
plt.ylabel('latitude')
plt.colorbar(location_plot, label='price_total')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('- The map shows listings grouped by city/location clusters rather than randomly spread points.')
print('- Price intensity changes across clusters, which reinforces the importance of geographic features.')
print('- Latitude and longitude are useful for exploration, but the modeling notebook uses city, district, and proximity variables for a more interpretable pre-listing model.')

### Numeric Correlations With Price

In [ ]:
top_numeric_correlations = numeric_correlations.head(12)
corr_features = ['price_total'] + top_numeric_correlations.index.tolist()

plt.figure(figsize=(12, 9))
sns.heatmap(eda_df[corr_features].corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation heatmap: price_total and top numeric variables')
plt.tight_layout()
plt.show()

display(top_numeric_correlations.to_frame('pearson_corr_with_price'))

strongest_numeric_corr = top_numeric_correlations.index[0]
strongest_numeric_corr_value = top_numeric_correlations.iloc[0]

print('Interpretation:')
print(f'- The strongest numeric correlation with price_total is {strongest_numeric_corr}: {strongest_numeric_corr_value:.3f}.')
print('- Location/tourism proximity variables are more informative than review-score variables for explaining price.')
print('- Correlation is only a linear relationship, so the modeling notebook also uses tree-based models to capture non-linear effects.')

## Feature Selection For Modeling

In [ ]:
available_numeric_features = [
    'max_guests', 'num_bedrooms', 'distance_city_center', 'distance_metro',
    'attraction_index', 'attraction_index_norm',
    'restaurant_index', 'restaurant_index_norm',
    'proximity_index', 'longitude', 'latitude',
]

target_correlations = (
    eda_df[available_numeric_features + ['price_total']]
    .corr(numeric_only=True)['price_total']
    .drop('price_total')
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .to_frame('pearson_corr_with_price')
)

numeric_corr = eda_df[available_numeric_features].corr(numeric_only=True).abs()
highly_correlated_pairs = []

for index, feature_1 in enumerate(available_numeric_features):
    for feature_2 in available_numeric_features[index + 1:]:
        corr_value = numeric_corr.loc[feature_1, feature_2]
        if corr_value >= 0.85:
            highly_correlated_pairs.append({
                'feature_1': feature_1,
                'feature_2': feature_2,
                'abs_corr': corr_value,
            })

print('Selected practical model features:')
print(MODEL_FEATURES)
print(f'Rare districts are grouped as Other when count < {DISTRICT_MIN_COUNT}. This reduces sparse one-hot columns and helps lower overfitting.')

print('\nExcluded feature groups:')
for reason, features in EXCLUDED_FEATURE_GROUPS.items():
    print(f'- {reason}: {features}')

print('\nHighly correlated practical numeric pairs:')
display(pd.DataFrame(highly_correlated_pairs))

print('\nPractical numeric correlations with price_total:')
display(target_correlations)

## Processed Dataset

In [ ]:
model_df, model_input_df, metadata = build_model_dataframe(raw_df)
processed_path = save_processed_data(model_df)

print(f'Processed dataset saved to: {processed_path}')
print(f'Processed shape: {model_df.shape}')
display(model_df.head())